In [1]:
import numpy as np
import pandas as pd
import ast
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pickle

In [2]:
# Load datasets
movies = pd.read_csv('data/movies.csv')
tags = pd.read_csv('data/tags.csv')
links = pd.read_csv('data/links.csv')

***Content-Based Filtering***

In [3]:
# Group tags by movieId
tags_grouped = tags.groupby('movieId')['tag'].apply(lambda x: ' '.join(x)).reset_index()

# Merge movies with tags
movies = movies.merge(tags_grouped, on='movieId', how='left')

# Merge movies with links
movies = movies.merge(links, on='movieId', how='left')

# Fill missing tags
movies['tag'] = movies['tag'].fillna('')

# Clean genres column
movies['genres'] = movies['genres'].apply(lambda x: x.replace('|', ' '))

# Create tags column by combining title + genres + user tags
movies['tags'] = movies['title'] + ' ' + movies['genres'] + ' ' + movies['tag']

# Final dataset with relevant columns
movies = movies[['movieId', 'title', 'genres', 'tags', 'tmdbId']]

# Convert tags to lowercase
movies['tags'] = movies['tags'].apply(lambda x: x.lower())

# Function to fix movie titles
def clean_title(title):
    # Move ", The" to front
    if ", The" in title:
        title = "The " + title.replace(", The", "")
    # Move ", A" to front
    elif ", A" in title:
        title = "A " + title.replace(", A", "")
    # Move ", An" to front
    elif ", An" in title:
        title = "An " + title.replace(", An", "")
    return title

movies['title'] = movies['title'].apply(clean_title)

In [4]:
movies.head()

,movieId,title,genres,tags,tmdbId
0,1,Toy Story (1995),Adventure Animation Children Comedy Fantasy,toy story (1995) adventure animation children ...,862.0
1,2,Jumanji (1995),Adventure Children Fantasy,jumanji (1995) adventure children fantasy fant...,8844.0
2,3,Grumpier Old Men (1995),Comedy Romance,grumpier old men (1995) comedy romance moldy old,15602.0
3,4,Waiting to Exhale (1995),Comedy Drama Romance,waiting to exhale (1995) comedy drama romance,31357.0
4,5,Father of the Bride Part II (1995),Comedy,father of the bride part ii (1995) comedy preg...,11862.0


In [5]:
movies['title'] = movies['title'].astype(object)
movies['tags'] = movies['tags'].astype(object)

if 'genres' in movies.columns:
    movies['genres'] = movies['genres'].astype(object)

movies['tmdbId'] = pd.to_numeric(movies['tmdbId'], errors='coerce')

In [6]:
# Stemming
ps = PorterStemmer()

def stemming(text):
    l = []
    for i in text.split():
        l.append(ps.stem(i))
    return " ".join(l)

movies['tags'] = movies['tags'].apply(stemming)

In [7]:
# Vectorization
vectorizer = CountVectorizer(max_features=5000, stop_words='english')
vectors = vectorizer.fit_transform(movies['tags']).toarray()

# Cosine similarity
content_similarity = cosine_similarity(vectors)

In [8]:
# Recommendation function
def recommendation_system(movie_title):
    movie_index = movies[movies['title'] == movie_title].index[0]
    distances = sorted(list(enumerate(content_similarity[movie_index])), reverse=True, key=lambda x: x[1])
    
    for i in distances[1:20]:
        print(movies.iloc[i[0]].title)

In [9]:
# Example usage
recommendation_system('Toy Story (1995)')

Toy Story 2 (1999)
Toy Story 3 (2010)
A Bug's Life (1998)
Antz (1998)
Moana (2016)
Balto (1995)
The Adventures of Rocky and Bullwinkle (2000)
We're Back! A Dinosaur's Story (1993)
Monsters, Inc. (2001)
Valiant (2005)
The Wild (2006)
Shrek the Third (2007)
The Tale of Despereaux (2008)
Turbo (2013)
The Indian in the Cupboard (1995)
Gordy (1995)
The Borrowers (1997)
Bedtime Stories (2008)
Presto (2008)


In [10]:
# Save content-based model files
pickle.dump(movies, open('movies.pkl', 'wb'))
pickle.dump(content_similarity, open('content_similarity.pkl', 'wb'))

***User-Based Collaborative Filtering***

In [11]:
ratings = pd.read_csv('data/ratings.csv')

# Merge ratings with movie titles
ratings_with_movies = ratings.merge(movies[['movieId', 'title']], on='movieId')
ratings_with_movies.head()

,userId,movieId,rating,timestamp,title
0,1,1,4.0,964982703,Toy Story (1995)
1,1,3,4.0,964981247,Grumpier Old Men (1995)
2,1,6,4.0,964982224,Heat (1995)
3,1,47,5.0,964983815,Seven (a.k.a. Se7en) (1995)
4,1,50,5.0,964982931,The Usual Suspects (1995)


In [12]:
# Create user-movie matrix
user_movie_matrix = ratings_with_movies.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

# Fill missing ratings with 0
user_movie_matrix_filled = user_movie_matrix.fillna(0)

In [13]:
# Merge ratings with movie titles
ratings_with_movies = ratings.merge(movies[['movieId', 'title']], on='movieId')

# Create user-movie matrix
user_movie_matrix = ratings_with_movies.pivot_table(
    index='userId',
    columns='title',
    values='rating'
)

# Fill missing ratings with 0
user_movie_matrix_filled = user_movie_matrix.fillna(0)

In [14]:
# User similarity using cosine similarity
user_similarity = cosine_similarity(user_movie_matrix_filled)

# Convert to dataframe for easier use
user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_movie_matrix.index,
    columns=user_movie_matrix.index
)

In [15]:
# User-based recommendation function
def user_based_recommendation_system(user_id):
    # Find similar users
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)

    # Remove the selected user
    similar_users = similar_users.drop(user_id)

    # Get top similar users
    top_users = similar_users.head(10)

    # Movies already watched by selected user
    watched_movies = user_movie_matrix.loc[user_id].dropna().index

    # Store recommendation scores
    recommendation_scores = {}

    # Loop through similar users
    for similar_user_id, similarity_score in top_users.items():
        similar_user_ratings = user_movie_matrix.loc[similar_user_id].dropna()

        for movie_title, rating in similar_user_ratings.items():
            if movie_title not in watched_movies:
                if movie_title not in recommendation_scores:
                    recommendation_scores[movie_title] = 0

                recommendation_scores[movie_title] += similarity_score * rating

    # Sort recommendations
    recommendations = sorted(
        recommendation_scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    # Print top 20 recommendations
    for movie, score in recommendations[:20]:
        print(movie)

In [16]:
# Example usage
user_based_recommendation_system(1)


Terminator 2: Judgment Day (1991)
Aliens (1986)
The Sixth Sense (1999)
The Hunt for Red October (1990)
The Godfather (1972)
2001: A Space Odyssey (1968)
Die Hard (1988)
Blade Runner (1982)
The Godfather: Part II (1974)
The Breakfast Club (1985)
Star Trek II: The Wrath of Khan (1982)
Gattaca (1997)
Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
Airplane! (1980)
The Fifth Element (1997)
Jaws (1975)
True Lies (1994)
Ferris Bueller's Day Off (1986)
Star Trek III: The Search for Spock (1984)
Raising Arizona (1987)


In [17]:
# Save user-based model files
pickle.dump(user_movie_matrix, open('user_movie_matrix.pkl', 'wb'))
pickle.dump(user_similarity_df, open('user_similarity.pkl', 'wb'))

***Hybrid Recommender Method***

In [18]:
def hybrid_recommendation_system(user_id, movie_title):

    # Get index of selected movie
    movie_index = movies[movies['title'] == movie_title].index[0]

    # Get content similarity scores for selected movie
    content_scores = list(enumerate(content_similarity[movie_index]))

    # Store content-based scores in dictionary
    content_score_dict = {}

    # Loop through all content similarity scores
    for i in content_scores:

        # Get movie title
        movie_name = movies.iloc[i[0]].title

        # Get similarity score
        score = i[1]

        # Store movie and score
        content_score_dict[movie_name] = score


    # Find users similar to selected user
    similar_users = user_similarity_df[user_id].sort_values(ascending=False)

    # Remove selected user from similarity list
    similar_users = similar_users.drop(user_id)

    # Get top 10 most similar users
    top_users = similar_users.head(10)

    # Get movies already watched by selected user
    watched_movies = user_movie_matrix.loc[user_id].dropna().index

    # Store collaborative filtering scores
    collaborative_score_dict = {}

    # Loop through top similar users
    for similar_user_id, similarity_score in top_users.items():

        # Get ratings from similar user
        similar_user_ratings = user_movie_matrix.loc[similar_user_id].dropna()

        # Loop through movies rated by similar user
        for movie_name, rating in similar_user_ratings.items():

            # Recommend only movies not already watched
            if movie_name not in watched_movies:

                # Create movie entry if not already in dictionary
                if movie_name not in collaborative_score_dict:
                    collaborative_score_dict[movie_name] = 0

                # Add weighted collaborative score
                collaborative_score_dict[movie_name] += similarity_score * rating


    # Find maximum collaborative score
    max_collaborative_score = max(collaborative_score_dict.values())

    # Normalise collaborative scores between 0 and 1
    for movie_name in collaborative_score_dict:

        collaborative_score_dict[movie_name] = (
            collaborative_score_dict[movie_name] / max_collaborative_score
        )


    # Store final hybrid scores
    final_scores = {}

    # Combine content-based and collaborative scores
    for movie_name in collaborative_score_dict:

        # Get content-based similarity score
        content_score = content_score_dict.get(movie_name, 0)

        # Get collaborative filtering score
        collaborative_score = collaborative_score_dict.get(movie_name, 0)

        # Hybrid score calculation
        final_score = ((0.5 * content_score) +(0.5 * collaborative_score))

        # Store final score
        final_scores[movie_name] = final_score


    # Sort recommendations by highest score
    recommendations = sorted(final_scores.items(), key=lambda x: x[1], reverse=True)

    # Print top 20 recommendations
    for movie, score in recommendations[:20]:
        print(movie)

In [19]:
hybrid_recommendation_system(1, 'Toy Story (1995)')

Toy Story 2 (1999)
A Bug's Life (1998)
The Hunt for Red October (1990)
Aliens (1986)
Terminator 2: Judgment Day (1991)
The Fifth Element (1997)
The Breakfast Club (1985)
Airplane! (1980)
2001: A Space Odyssey (1968)
The Sixth Sense (1999)
True Lies (1994)
The Godfather (1972)
Star Trek II: The Wrath of Khan (1982)
Aladdin (1992)
Die Hard (1988)
Army of Darkness (1993)
Blade Runner (1982)
The Godfather: Part II (1974)
Ferris Bueller's Day Off (1986)
Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
